# 01_load_data.py

---------------
Downloads and caches FastF1 data for ALL drivers across the full 2025 F1
season (Race sessions only).

Outputs
-------
  data/laps.csv             – per-lap data (enhanced) + weather merged by time
  data/telemetry.csv        – raw telemetry for all laps
  data/telemetry_labelled.csv – telemetry + Is_Anomaly column
  data/weather.csv          – raw weather time-series per round
  data/results.csv          – race results + driver info per round

What FastF1 does NOT provide (not in the API):
  - Car setups (confidential, team-only)
  - Tyre temperatures / pressures from sensors
  - Fuel loads
  - Engine modes

Runtime: several hours on first run; fast on re-runs thanks to the local cache.

In [ ]:
import os
import fastf1
import pandas as pd
import numpy as np

## Paths

In [ ]:
CACHE_DIR = "../cache"
DATA_DIR  = "../data"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(DATA_DIR,  exist_ok=True)

fastf1.Cache.enable_cache(CACHE_DIR)

YEAR = 2025

## Column definitions

In [ ]:
LAP_COLS = [
    # Identity
    "Driver", "DriverNumber", "Team",
    # Timing
    "LapNumber", "LapTime",
    "Sector1Time", "Sector2Time", "Sector3Time",
    "Sector1SessionTime", "Sector2SessionTime", "Sector3SessionTime",
    "LapStartTime", "LapStartDate",
    # Speed traps
    "SpeedI1", "SpeedI2", "SpeedFL", "SpeedST",
    # Tyre
    "Compound", "TyreLife", "FreshTyre", "Stint",
    # Pit
    "PitOutTime", "PitInTime",
    # Race position & status
    "Position",
    "TrackStatus",
    "IsPersonalBest",
    # Data quality flags
    "IsAccurate", "Deleted", "DeletedReason",
]

TIMEDELTA_COLS = [
    "LapTime",
    "Sector1Time", "Sector2Time", "Sector3Time",
    "Sector1SessionTime", "Sector2SessionTime", "Sector3SessionTime",
    "LapStartTime", "PitOutTime", "PitInTime",
]

TEL_COLS = ["Time", "Speed", "Throttle", "Brake", "RPM", "nGear", "DRS",
            "X", "Y", "Z"]

WEATHER_COLS = ["Time", "AirTemp", "Humidity", "Pressure",
                "Rainfall", "TrackTemp", "WindDirection", "WindSpeed"]

RESULT_COLS = [
    "DriverNumber", "Abbreviation", "FullName", "TeamName", "TeamId",
    "GridPosition", "Position", "ClassifiedPosition",
    "Points", "Status", "Laps",
    "Q1", "Q2", "Q3", "Time",
    "CountryCode",
]

## Discover all 2025 race rounds

In [ ]:
print(f"Fetching {YEAR} event schedule …")
schedule = fastf1.get_event_schedule(YEAR, include_testing=False)
race_rounds = schedule[schedule["EventFormat"] != "testing"]["RoundNumber"].tolist()
print(f"  Found {len(race_rounds)} race rounds: {race_rounds}\n")

## Accumulators

In [ ]:
all_laps        = []
all_telemetry   = []
all_weather     = []
all_results     = []
all_race_ctrl   = []

## Main loop

In [ ]:
for rnd in race_rounds:
    event_name = schedule.loc[
        schedule["RoundNumber"] == rnd, "EventName"
    ].values[0]
    circuit    = schedule.loc[schedule["RoundNumber"] == rnd, "Location"].values[0]
    country    = schedule.loc[schedule["RoundNumber"] == rnd, "Country"].values[0]

    print(f"[Round {rnd:02d}] {event_name}  ({circuit}, {country}) …")

    try:
        session = fastf1.get_session(YEAR, rnd, "R")
        session.load(telemetry=True, laps=True, weather=True, messages=True)
    except Exception as e:
        print(f"  ⚠  Could not load session: {e}")
        continue

    # ── Race results & driver info ────────────────────────────────────────────
    try:
        res = session.results
        if res is not None and not res.empty:
            res_df = res[[c for c in RESULT_COLS if c in res.columns]].copy()
            # Q1/Q2/Q3 and Time are timedeltas
            for tc in ["Q1", "Q2", "Q3", "Time"]:
                if tc in res_df.columns:
                    res_df[tc] = res_df[tc].apply(
                        lambda x: x.total_seconds() if pd.notna(x) and hasattr(x, "total_seconds") else np.nan
                    )
            res_df.insert(0, "Round",     rnd)
            res_df.insert(1, "EventName", event_name)
            res_df.insert(2, "Circuit",   circuit)
            res_df.insert(3, "Country",   country)
            all_results.append(res_df)
            print(f"  Results: {len(res_df)} drivers")
    except Exception as e:
        print(f"  ⚠  Results error: {e}")

    # ── Weather ───────────────────────────────────────────────────────────────
    weather_available = False
    weather_df_raw    = None
    try:
        wd = session.weather_data
        if wd is not None and not wd.empty:
            weather_df_raw = wd[[c for c in WEATHER_COLS if c in wd.columns]].copy()
            weather_df_raw["Time"] = weather_df_raw["Time"].dt.total_seconds()
            weather_df_raw.insert(0, "Round",     rnd)
            weather_df_raw.insert(1, "EventName", event_name)
            all_weather.append(weather_df_raw.copy())
            weather_available = True
            print(f"  Weather rows: {len(weather_df_raw)}")
    except Exception as e:
        print(f"  ⚠  Weather error: {e}")

    # ── Race control messages ─────────────────────────────────────────────────
    try:
        rc = session.race_control_messages
        if rc is not None and not rc.empty:
            rc_df = rc.copy()
            rc_df["Time"] = rc_df["Time"].dt.total_seconds()
            rc_df.insert(0, "Round",     rnd)
            rc_df.insert(1, "EventName", event_name)
            all_race_ctrl.append(rc_df)
            print(f"  Race control messages: {len(rc_df)}")
    except Exception as e:
        print(f"  ⚠  Race control error: {e}")

    # ── Laps ──────────────────────────────────────────────────────────────────
    all_driver_laps = session.laps.reset_index(drop=True)
    if all_driver_laps.empty:
        print(f"  ⚠  No laps found for this session")
        continue

    drivers = all_driver_laps["Driver"].unique()
    print(f"  Drivers: {sorted(drivers)}")

    laps_df = all_driver_laps[[c for c in LAP_COLS if c in all_driver_laps.columns]].copy()

    for col in TIMEDELTA_COLS:
        if col in laps_df.columns:
            laps_df[col] = laps_df[col].dt.total_seconds()

    laps_df.insert(0, "Round",     rnd)
    laps_df.insert(1, "EventName", event_name)
    laps_df.insert(2, "Circuit",   circuit)
    laps_df.insert(3, "Country",   country)

    # ── Merge weather into laps by nearest session time ───────────────────────
    # LapStartTime is seconds from session start; weather Time is the same axis.
    if weather_available and weather_df_raw is not None and "LapStartTime" in laps_df.columns:
        try:
            weather_cols_to_merge = [c for c in WEATHER_COLS
                                     if c != "Time" and c in weather_df_raw.columns]
            weather_for_merge = (
                weather_df_raw[["Time"] + weather_cols_to_merge]
                .sort_values("Time")
                .reset_index(drop=True)
            )
            laps_sorted = laps_df.sort_values("LapStartTime").reset_index()
            merged = pd.merge_asof(
                laps_sorted,
                weather_for_merge.rename(columns={"Time": "LapStartTime"}),
                on="LapStartTime",
                direction="nearest",
            )
            laps_df = merged.set_index("index").sort_index().reset_index(drop=True)
        except Exception as e:
            print(f"  ⚠  Weather merge error: {e}")

    all_laps.append(laps_df)
    print(f"  Laps: {len(laps_df)}")

    # ── Telemetry ─────────────────────────────────────────────────────────────
    race_tel_frames = []
    for drv in drivers:
        driver_laps = all_driver_laps[all_driver_laps["Driver"] == drv]
        drv_count   = 0
        for _, lap in driver_laps.iterrows():
            try:
                tel   = lap.get_telemetry()
                avail = [c for c in TEL_COLS if c in tel.columns]
                tel   = tel[avail].copy()
                tel["Driver"]    = drv
                tel["LapNumber"] = lap["LapNumber"]
                tel["LapTime_s"] = (
                    lap["LapTime"].total_seconds()
                    if pd.notna(lap["LapTime"]) else np.nan
                )
                race_tel_frames.append(tel)
                drv_count += 1
            except Exception:
                pass
        if drv_count == 0:
            print(f"    ⚠  No telemetry for {drv}")

    if race_tel_frames:
        race_tel = pd.concat(race_tel_frames, ignore_index=True)
        if "Time" in race_tel.columns:
            race_tel["Time"] = race_tel["Time"].dt.total_seconds()
        race_tel.insert(0, "Round",     rnd)
        race_tel.insert(1, "EventName", event_name)
        all_telemetry.append(race_tel)
        print(f"  Telemetry rows: {len(race_tel):,}")
    else:
        print(f"  ⚠  No telemetry extracted for this round")

## Concatenate & save

In [ ]:
print("\nConcatenating and saving …")

laps_all = pd.concat(all_laps, ignore_index=True)
laps_all.to_csv(f"{DATA_DIR}/laps.csv", index=False)
print(f"  laps.csv            {len(laps_all):,} rows | "
      f"{laps_all['Round'].nunique()} rounds | {laps_all['Driver'].nunique()} drivers | "
      f"{len(laps_all.columns)} columns")

telemetry_all = pd.concat(all_telemetry, ignore_index=True)
telemetry_all.to_csv(f"{DATA_DIR}/telemetry.csv", index=False)
print(f"  telemetry.csv       {len(telemetry_all):,} rows")

if all_weather:
    weather_all = pd.concat(all_weather, ignore_index=True)
    weather_all.to_csv(f"{DATA_DIR}/weather.csv", index=False)
    print(f"  weather.csv         {len(weather_all):,} rows")

if all_results:
    results_all = pd.concat(all_results, ignore_index=True)
    results_all.to_csv(f"{DATA_DIR}/results.csv", index=False)
    print(f"  results.csv         {len(results_all):,} rows | "
          f"{len(results_all.columns)} columns")

if all_race_ctrl:
    rc_all = pd.concat(all_race_ctrl, ignore_index=True)
    rc_all.to_csv(f"{DATA_DIR}/race_control.csv", index=False)
    print(f"  race_control.csv    {len(rc_all):,} rows")

## Anomaly labelling

In [ ]:
# A lap is anomalous if:
#   (a) LapTime > driver_race_mean + 1.5 * std, OR
#   (b) TrackStatus != '1'  (safety car, VSC, red flag), OR
#   (c) Deleted == True     (track limits violation, etc.)
print("\nLabelling anomalies …")

anomaly_keys = set()  # (Round, Driver, LapNumber)

for (rnd, drv), grp in laps_all.groupby(["Round", "Driver"]):
    lap_times = grp["LapTime"].dropna()
    if len(lap_times) >= 3:
        thr = lap_times.mean() + 1.5 * lap_times.std()
        for ln in grp.loc[grp["LapTime"] > thr, "LapNumber"]:
            anomaly_keys.add((rnd, drv, ln))

    if "TrackStatus" in grp.columns:
        for ln in grp.loc[grp["TrackStatus"] != "1", "LapNumber"]:
            anomaly_keys.add((rnd, drv, ln))

    if "Deleted" in grp.columns:
        for ln in grp.loc[grp["Deleted"] == True, "LapNumber"]:
            anomaly_keys.add((rnd, drv, ln))

telemetry_labelled = telemetry_all.copy()
telemetry_labelled["Is_Anomaly"] = telemetry_labelled.apply(
    lambda row: int((row["Round"], row["Driver"], row["LapNumber"]) in anomaly_keys),
    axis=1,
)

n_anomaly = int(telemetry_labelled["Is_Anomaly"].sum())
n_total   = len(telemetry_labelled)
print(f"  Anomaly telemetry rows: {n_anomaly:,} / {n_total:,} "
      f"({100 * n_anomaly / n_total:.2f} %)")

telemetry_labelled.to_csv(f"{DATA_DIR}/telemetry_labelled.csv", index=False)
print(f"  telemetry_labelled.csv  saved")

## Season summary

In [ ]:
print("\n── Season summary (laps per driver) ─────────────────────────────────")
summary = (
    laps_all.groupby("Driver")
    .agg(
        Rounds      =("Round",    "nunique"),
        TotalLaps   =("LapNumber","count"),
        AvgLapTime_s=("LapTime",  "mean"),
    )
    .sort_values("TotalLaps", ascending=False)
    .reset_index()
)
print(summary.to_string(index=False))

print("\n── Laps per round ───────────────────────────────────────────────────")
round_summary = (
    laps_all.groupby(["Round", "EventName", "Circuit", "Country"])
    .agg(
        Drivers  =("Driver",    "nunique"),
        TotalLaps=("LapNumber", "count"),
    )
    .reset_index()
)
print(round_summary.to_string(index=False))

print("\n── Lap columns saved ───────────────────────────────────────────────")
print(sorted(laps_all.columns.tolist()))

print("\n✅  01_load_data.py complete.")